In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

In [ ]:
BASE_DIR = Path(".")
PROCESSED_DIR = BASE_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(
    PROCESSED_DIR / "eda_dataset.csv"
)

df["date"] = pd.to_datetime(df["date"])

In [ ]:
df.shape

(840, 24)

In [ ]:
df.head()

,date,estate_id,tbs_production_ton,cpo_production_ton,kernel_production_ton,tbs_outlier_flag,cpo_extraction_rate,kernel_extraction_rate,rainfall_mm,rainy_days,...,region,area_ha,planting_year,plant_age_years,tbs_yield_ton_per_ha,month,month_name,year,rainfall_category,fertilizer_kg_per_ha
0,2019-01-01,EST001,8132.4,1600.4,487.9,False,0.196793,0.059995,265.4,18.0,...,Sumatra,4250,2010,9,1.913506,1,Jan,2019,Optimal,10.484235
1,2019-02-01,EST001,8492.3,1805.1,443.2,False,0.212557,0.052188,238.5,17.0,...,Sumatra,4250,2010,9,1.998188,2,Feb,2019,Optimal,12.072000
2,2019-03-01,EST001,8186.5,1597.9,413.1,False,0.195187,0.050461,247.2,15.0,...,Sumatra,4250,2010,9,1.926235,3,Mar,2019,Optimal,10.342118
3,2019-04-01,EST001,7936.4,1398.5,415.7,False,0.176213,0.052379,247.1,17.0,...,Sumatra,4250,2010,9,1.867388,4,Apr,2019,Optimal,8.566118
4,2019-05-01,EST001,8551.4,1856.5,513.1,False,0.217099,0.060002,192.1,13.0,...,Sumatra,4250,2010,9,2.012094,5,May,2019,Optimal,9.754118


## Sort Data

In [ ]:
df = df.sort_values(
    ["estate_id", "date"]
).reset_index(drop=True)

## Time Features

In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter

In [ ]:
df[
    ["date", "year", "month", "quarter"]
].head(15)

,date,year,month,quarter
0,2019-01-01,2019,1,1
1,2019-02-01,2019,2,1
2,2019-03-01,2019,3,1
3,2019-04-01,2019,4,2
4,2019-05-01,2019,5,2
5,2019-06-01,2019,6,2
6,2019-07-01,2019,7,3
7,2019-08-01,2019,8,3
8,2019-09-01,2019,9,3
9,2019-10-01,2019,10,4


## Cyclical Month Features

> masalahnya: desember = 12 dan januari = 1, Karena itu kita representasikan bulan sebagai siklus. Ini nantinya bisa membantu model machine learning memahami pola musiman.



In [ ]:
df["month_sin"] = np.sin(
    2 * np.pi * df["month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["month"] / 12
)

In [ ]:
df[
    [
        "month",
        "month_sin",
        "month_cos"
    ]
].drop_duplicates().sort_values("month")

,month,month_sin,month_cos
0,1,5.000000e-01,8.660254e-01
1,2,8.660254e-01,5.000000e-01
2,3,1.000000e+00,6.123234e-17
3,4,8.660254e-01,-5.000000e-01
4,5,5.000000e-01,-8.660254e-01
5,6,1.224647e-16,-1.000000e+00
6,7,-5.000000e-01,-8.660254e-01
7,8,-8.660254e-01,-5.000000e-01
8,9,-1.000000e+00,-1.836970e-16
9,10,-8.660254e-01,5.000000e-01


## Plant Age Feature

In [ ]:
df["plant_age_years"] = (
    df["year"] - df["planting_year"]
)

In [ ]:
df["plant_age_group"] = pd.cut(
    df["plant_age_years"],
    bins=[0, 5, 9, 18, 22, np.inf],
    labels=[
        "Young",
        "Immature",
        "Prime",
        "Mature",
        "Old"
    ]
)

## Fertilizer Intensity

In [ ]:
df["fertilizer_kg_per_ha"] = (
    df["fertilizer_kg"] /
    df["area_ha"]
)

## Rainfall Lag Features

In [ ]:
# rainfall lag 1 bulan
df["rainfall_lag_1"] = (
    df.groupby("estate_id")["rainfall_mm"]
      .shift(1)
)

In [ ]:
# rainfall lag 3 bulan
df["rainfall_lag_3"] = (
    df.groupby("estate_id")["rainfall_mm"]
      .shift(3)
)

## Rainfall Rolling Average

In [ ]:
# 3-month rainfall average
df["rainfall_3m_avg"] = (
    df.groupby("estate_id")["rainfall_mm"]
      .transform(
          lambda x: x.rolling(3).mean()
      )
)

In [ ]:
# 6-month rainfall average
df["rainfall_6m_avg"] = (
    df.groupby("estate_id")["rainfall_mm"]
      .transform(
          lambda x: x.rolling(6).mean()
      )
)

## TBS Lag Features

> Jadi misalnya: tbs_lag_1, berarti: produksi TBS bulan sebelumnya. Sedangkan: tbs_lag_12, berarti: produksi TBS pada bulan yang sama tahun sebelumnya.

In [ ]:
# lag 1
df["tbs_lag_1"] = (
    df.groupby("estate_id")["tbs_production_ton"]
      .shift(1)
)

In [ ]:
# lag 3
df["tbs_lag_3"] = (
    df.groupby("estate_id")["tbs_production_ton"]
      .shift(3)
)

In [ ]:
# lag 6
df["tbs_lag_6"] = (
    df.groupby("estate_id")["tbs_production_ton"]
      .shift(6)
)

In [ ]:
# lag 12
df["tbs_lag_12"] = (
    df.groupby("estate_id")["tbs_production_ton"]
      .shift(12)
)

## Rolling TBS Features

In [ ]:
# 3 bulan
df["tbs_rolling_mean_3"] = (
    df.groupby("estate_id")["tbs_production_ton"]
      .transform(
          lambda x: x.shift(1).rolling(3).mean()
      )
)

In [ ]:
# 6 bulan
df["tbs_rolling_mean_6"] = (
    df.groupby("estate_id")["tbs_production_ton"]
      .transform(
          lambda x: x.shift(1).rolling(6).mean()
      )
)

In [ ]:
# 12 bulan
df["tbs_rolling_mean_12"] = (
    df.groupby("estate_id")["tbs_production_ton"]
      .transform(
          lambda x: x.shift(1).rolling(12).mean()
      )
)

## Production Growth Features

In [ ]:
# perubahan produksi dibanding bulan sebelumnya
df["tbs_growth_1m"] = (
    df.groupby("estate_id")["tbs_production_ton"]
      .pct_change(1)
)

In [ ]:
# year over year
df["tbs_growth_12m"] = (
    df.groupby("estate_id")["tbs_production_ton"]
      .pct_change(12)
)

## Candidate Feature List

In [ ]:
features = [
    # Time
    "month",
    "quarter",
    "month_sin",
    "month_cos",

    # Estate / Agronomy
    "area_ha",
    "plant_age_years",
    "fertilizer_kg_per_ha",

    # Weather
    "rainfall_mm",
    "rainy_days",
    "rainfall_lag_1",
    "rainfall_lag_3",
    "rainfall_3m_avg",
    "rainfall_6m_avg",

    # Operations
    "working_days",
    "disruption_days",

    # Historical production
    "tbs_lag_1",
    "tbs_lag_3",
    "tbs_lag_6",
    "tbs_lag_12",
    "tbs_rolling_mean_3",
    "tbs_rolling_mean_6",
    "tbs_rolling_mean_12",
    "tbs_growth_1m",
    "tbs_growth_12m"
]

target = "tbs_production_ton"

## Cek Missing Values Akibat Lag

In [ ]:
df[features].isna().sum().sort_values(
    ascending=False
)

,0
tbs_growth_12m,120
tbs_lag_12,120
tbs_rolling_mean_12,120
tbs_lag_6,60
tbs_rolling_mean_6,60
rainfall_6m_avg,50
rainfall_lag_3,30
tbs_lag_3,30
tbs_rolling_mean_3,30
rainfall_3m_avg,20


## Buat Modeling Dataset

In [ ]:
model_df = df[
    ["date", "estate_id", "estate_name", "region"]
    + features
    + [target]
].copy()

In [ ]:
model_df = model_df.dropna(
    subset=features + [target]
).reset_index(drop=True)

In [ ]:
model_df.shape

(720, 29)

In [ ]:
model_df.head()

,date,estate_id,estate_name,region,month,quarter,month_sin,month_cos,area_ha,plant_age_years,...,tbs_lag_1,tbs_lag_3,tbs_lag_6,tbs_lag_12,tbs_rolling_mean_3,tbs_rolling_mean_6,tbs_rolling_mean_12,tbs_growth_1m,tbs_growth_12m,tbs_production_ton
0,2020-01-01,EST001,Estate Lampung 01,Sumatra,1,1,0.500000,8.660254e-01,4250,10,...,8680.5,7933.2,9134.3,8132.4,8177.300000,8281.833333,8307.600000,-0.041749,0.022835,8318.1
1,2020-02-01,EST001,Estate Lampung 01,Sumatra,2,1,0.866025,5.000000e-01,4250,10,...,8318.1,7918.2,7770.7,8492.3,8305.600000,8145.800000,8323.075000,0.016350,-0.004498,8454.1
2,2020-03-01,EST001,Estate Lampung 01,Sumatra,3,1,1.000000,6.123234e-17,4250,10,...,8454.1,8680.5,8254.1,8186.5,8484.233333,8259.700000,8319.891667,0.151690,0.189336,9736.5
3,2020-04-01,EST001,Estate Lampung 01,Sumatra,4,2,0.866025,-5.000000e-01,4250,10,...,9736.5,8318.1,7933.2,7936.4,8836.233333,8506.766667,8449.058333,0.036687,0.271823,10093.7
4,2020-05-01,EST001,Estate Lampung 01,Sumatra,5,2,0.500000,-8.660254e-01,4250,10,...,10093.7,8454.1,7918.2,8551.4,9428.100000,8866.850000,8628.833333,-0.135154,0.020827,8729.5


## Time-Based Train/Test Split

In [ ]:
train_df = model_df[
    model_df["date"] < "2025-01-01"
].copy()

test_df = model_df[
    model_df["date"] >= "2025-01-01"
].copy()

In [ ]:
print("Train:", train_df["date"].min(), "→", train_df["date"].max())
print("Test :", test_df["date"].min(), "→", test_df["date"].max())

Train: 2020-01-01 00:00:00 → 2024-12-01 00:00:00
Test : 2025-01-01 00:00:00 → 2025-12-01 00:00:00


In [ ]:
print("Train rows:", len(train_df))
print("Test rows :", len(test_df))

Train rows: 600
Test rows : 120


## Prepare X dan y

In [ ]:
X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

In [ ]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (600, 24)
y_train: (600,)
X_test: (120, 24)
y_test: (120,)


## Feature Summary

In [ ]:
feature_summary = pd.DataFrame({
    "feature": features,
    "category": [
        "Time",
        "Time",
        "Time",
        "Time",

        "Agronomy",
        "Agronomy",
        "Agronomy",

        "Weather",
        "Weather",
        "Weather",
        "Weather",
        "Weather",
        "Weather",

        "Operations",
        "Operations",

        "Historical Production",
        "Historical Production",
        "Historical Production",
        "Historical Production",
        "Historical Production",
        "Historical Production",
        "Historical Production",
        "Historical Production",
        "Historical Production"
    ]
})

feature_summary

,feature,category
0,month,Time
1,quarter,Time
2,month_sin,Time
3,month_cos,Time
4,area_ha,Agronomy
5,plant_age_years,Agronomy
6,fertilizer_kg_per_ha,Agronomy
7,rainfall_mm,Weather
8,rainy_days,Weather
9,rainfall_lag_1,Weather


## Save Modeling Dataset

In [ ]:
model_df.to_csv(
    PROCESSED_DIR / "forecasting_dataset.csv",
    index=False
)

In [ ]:
train_df.to_csv(
    PROCESSED_DIR / "train_dataset.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_DIR / "test_dataset.csv",
    index=False
)